# Prodigy_Tuning_R28 — Verificare se Prodigy è il bottleneck

## Obiettivo

R27 ha mostrato che il "T_corr 0.5 di B1" era **illusione cross-driver quasi totale** (T_intra ≤ 0.058 ovunque) e rank-collapse severo (rank ~1.3 / 5). R28 verifica se Prodigy stesso è coinvolto, testando i tre **fix non-controversi noti dalla letteratura** (konstmish, paper Mishchenko & Defazio 2024):
1. **`d0=1e-5`** — fix konstmish Issue #27 (mai testato post bug-fix 2026-06-03)
2. **Budget step 3×** (`max_steps_per_epoch=300`) — Prodigy paper usa 10k+ step; noi 1000
3. **Combo A1+B1** — verifica se sblocco congiunto
4. **TRUE escape mechanism**: cosine warm restart `T0=5` (CosineAnnealingWarmRestarts) + A1 + B1

## Architettura

Mirror struttura `Prodigy_Fusion_Study_R26.ipynb` (8 celle, validate su Azure) + lezioni R27 (GLOBALS in Cell 1, idempotenza per-run, stream stdout subprocess, push per-run).

## Sweep

Tutti i run su base R25_B1 (champion R25): `lambda_T_aux=0.1`, `lambda_sr=0.5`, `max_delay=6`, `bit_shift=3`, `mixed scenario`. Output `results/Prodigy_Study/ProdigyTuning_R28/<axis>/<tag>/`.

| Tag | `d0` | `max_steps/ep` | `epochs` | scheduler | Sotto-cartella |
|---|---:|---:|---:|---|---|
| R28_A0_baseline_B1 | 1e-6 | 100 | 10 | cosine_no_restart | A_d0/ |
| R28_A1_d0_1e-5 | **1e-5** | 100 | 10 | cosine_no_restart | A_d0/ |
| R28_B1_steps_300 | 1e-6 | **300** | 10 | cosine_no_restart | B_steps/ |
| R28_C1_d0_steps_combo | **1e-5** | **300** | 10 | cosine_no_restart | C_combo/ |
| R28_D1_warm_restart | **1e-5** | **300** | 10 | **cosine (warm restart T0=5)** | D_warm_restart/ |

## Decisioni che escono
- `prodigy_d` finale: passa da 0.02-0.06 a >0.5 (sblocco) o resta stuck low?
- `prodigy_lr_eff` ultima epoca: resta sano (>0.001) o collassa a 0?
- T_intra_corr (post-R27 metric) migliora (>0.10) o resta degenere?
- rank_effective: passa da 1 a ≥3 (sblocco) o resta a 1?
- D1 vs C1: warm restart sblocca rispetto a no_restart?

In [ ]:
# Cell 1 -- Bootstrap + GLOBALS + ENV check
import sys, os, subprocess
import importlib.util as _imu

# === GLOBALS (R27 lesson: visibili a TUTTE le celle successive) ===
RESULTS_DIR = 'results/Prodigy_Study/ProdigyTuning_R28'
AGGREGATE_CSV = f'{RESULTS_DIR}/_aggregate_R28.csv'
BRANCH = 'Prodigy_Deep_Study'
_TMP_MSG = '/tmp/r28_msg.txt' if os.path.isdir('/tmp') else 'r28_msg.txt'

os.makedirs(RESULTS_DIR, exist_ok=True)

for pkg in ['prodigyopt', 'pandas', 'matplotlib']:
    if _imu.find_spec(pkg) is None:
        print(f'  installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

for f in ['train.py', 'core/network.py',
          'results/Prodigy_Study/Audit_R27/audit_summary.csv',
          'results/Prodigy_Study/Ablation_R25/B_loss/R25_B1_T_aux_low/config_snapshot.json']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

sys.path.insert(0, '.')
for mod in ['train','core.network']:
    if mod in sys.modules: del sys.modules[mod]

# Verifica R25 + R27 changes attive (val_T_tracking_corr + val_T_intra_corr)
from train import pinn_loss, CSVLogger, BatchCSVLogger
for c in ['val_T_tracking_corr', 'val_T_intra_corr', 'val_T_pred_mean']:
    assert c in CSVLogger.COLS, f'MISSING col: {c}'
print('  [OK] CSVLogger ha val_T_tracking_corr + val_T_intra_corr (R27)')
for c in ['gn_out_fc_T', 'gn_decoded_T', 'grad_dir_T']:
    assert c in BatchCSVLogger.COLS, f'MISSING col: {c}'
print('  [OK] BatchCSVLogger ha gradient per-canale (R25)')

import inspect
assert 'lam_T_aux' in inspect.signature(pinn_loss).parameters
help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--lambda_T_aux', '--cf_max_delay', '--prodigy_d0', '--T0', '--scheduler']:
    assert flag in help_txt, f'MISSING CLI: {flag}'
print('  [OK] CLI: prodigy_d0 + T0 + scheduler disponibili')

# Verifica scheduler choices (necessario per D1)
assert 'cosine_no_restart' in help_txt and 'cosine' in help_txt
print('  [OK] scheduler choices include cosine + cosine_no_restart')

br = subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
print(f'\n[GIT] branch: {br}')
assert br == BRANCH, f'Branch errato: {br} (atteso {BRANCH})'

print(f'\n[GLOBALS]')
print(f'  RESULTS_DIR  = {RESULTS_DIR}')
print(f'  AGGREGATE_CSV = {AGGREGATE_CSV}')
print(f'  BRANCH       = {BRANCH}')
print('\nENV check passed.')

In [ ]:
# Cell 2 -- EXPERIMENTS (5 run: A0 baseline, A1 d0, B1 steps, C1 combo, D1 warm restart)

# Config base derivata da R25_B1 (champion R25). NON modificare qui senza ragione.
COMMON = {
    'optimizer': 'prodigy', 'lr': 1.0,
    'd_coef': 1.0, 'growth_rate': 'inf',
    'epochs': 10, 'max_steps_per_epoch': 100,
    'seq_len': 50,
    'hidden_size': 32, 'rank': 8,
    'max_delay': 6, 'bit_shift': 3,
    'lambda_sr': 0.5, 'lambda_T_aux': 0.1,   # = B1 setup
    'scenario_mix': 'highway:0.4,urban:0.3,truck:0.2,mixed:0.1',
    'cut_in_ratio': 0.0,
    'scheduler': 'cosine_no_restart',
    'T0': 5,                                 # ignorato per cosine_no_restart, usato per D1 (cosine)
    'cache_path': 'data/cache_1500_mixed_cut0.0_ou0.0.pt',
}

def _exp(tag, desc, axis, **overrides):
    e = {**COMMON, 'tag': tag, 'desc': desc, 'axis': axis}
    e.update(overrides)
    return e

EXPERIMENTS = [
    # ── Asse A: d0 (fix konstmish Issue #27) ──
    _exp('R28_A0_baseline_B1', 'B1 replica (sanity check post R27)',
         'A_d0', d0=1e-6),
    _exp('R28_A1_d0_1e-5', 'd0=1e-5 (fix konstmish Issue #27)',
         'A_d0', d0=1e-5),
    # ── Asse B: step budget ──
    _exp('R28_B1_steps_300', 'max_steps=300 (3x budget Prodigy)',
         'B_steps', d0=1e-6, max_steps_per_epoch=300),
    # ── Asse C: stack A1+B1 ──
    _exp('R28_C1_d0_steps_combo', 'd0=1e-5 + max_steps=300 (stack A1+B1)',
         'C_combo', d0=1e-5, max_steps_per_epoch=300),
    # ── Asse D: TRUE escape mechanism (cosine warm restart T0=5) ──
    _exp('R28_D1_warm_restart', 'cosine warm restart T0=5 + d0=1e-5 + steps=300',
         'D_warm_restart', d0=1e-5, max_steps_per_epoch=300, scheduler='cosine'),
]

print(f'EXPERIMENTS: {len(EXPERIMENTS)}')
for e in EXPERIMENTS:
    print(f"  [{e['axis']:<16}] {e['tag']:<28} "
          f"d0={e['d0']:<6} steps={e['max_steps_per_epoch']:<3} "
          f"sched={e['scheduler']:<18}  -- {e['desc']}")

In [ ]:
# Cell 3 -- Cache check (riuso da R25 B1 — mixed scenario)
import os
cache = COMMON['cache_path']
if os.path.isfile(cache):
    sz_mb = os.path.getsize(cache) / 1e6
    print(f'  [OK] cache mixed esistente: {cache}  ({sz_mb:.1f} MB)')
    print(f'       -> Verra\' riusata da tutti i 5 run')
else:
    print(f'  [INFO] cache mancante: {cache}')
    print(f'         Verra\' generata dal primo run (~2-5 min CPU Azure)')
    print(f'         Successivi run riusano la stessa cache.')

In [ ]:
# Cell 4 -- PRE-FLIGHT: statica (param count) + smoke train.py 1ep × 3step
import torch, time, shutil, sys, subprocess
sys.path.insert(0, '.')
if 'core.network' in sys.modules: del sys.modules['core.network']
from core.network import CF_FSNN_Net

# 1) Static check: param count B1
torch.manual_seed(42)
m = CF_FSNN_Net(hidden_size=32, rank=8, max_delay=6, bit_shift=3)
n_params = sum(p.numel() for p in m.parameters())
assert n_params == 864, f'Param count mismatch: {n_params} (atteso 864 per B1)'
print(f'  [OK] CF_FSNN_Net B1: {n_params} param')

# 2) Smoke train.py 1ep x 3 step con A1 config (d0=1e-5)
tag_smoke = f'_R28_PREFLIGHT_{int(time.time())}'   # NFS-safe (timestamp univoco)
cmd_smoke = [sys.executable, 'train.py',
    '--training_method', 'baseline',
    '--epochs', '1', '--max_steps_per_epoch', '3',
    '--batch_size', '8', '--val_batch_size', '32',
    '--seq_len', '50',
    '--cf_hidden_size', '32', '--cf_rank', '8',
    '--cf_max_delay', '6', '--cf_bit_shift', '3',
    '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
    '--lambda_bc', '1.0', '--lambda_sr', '0.5', '--lambda_T_aux', '0.1',
    '--scenario_mix', COMMON['scenario_mix'],
    '--cut_in_ratio', '0.0', '--noise_scale', '0.0',
    '--n_train', '80', '--n_val', '40',
    '--optimizer', 'prodigy', '--lr', '1.0',
    '--scheduler', 'cosine_no_restart',
    '--prodigy_betas', '0.9,0.99', '--prodigy_d_coef', '1.0',
    '--prodigy_d0', '1e-5',           # ← test A1 here
    '--prodigy_weight_decay', '0.01',
    '--prodigy_use_bias_correction', '1', '--prodigy_safeguard_warmup', '1',
    '--prodigy_growth_rate', 'inf',
    '--max_inf_streak', '99999', '--early_stop_patience', '0',
    '--tag', tag_smoke]
print(f'\n  Running smoke: --prodigy_d0 1e-5, --epochs 1, --max_steps 3')
r = subprocess.run(cmd_smoke, capture_output=True, text=True, encoding='utf-8', errors='replace')
if r.returncode != 0:
    print('STDOUT:', r.stdout[-1500:])
    print('STDERR:', r.stderr[-1500:])
    raise RuntimeError(f'preflight smoke failed (exit {r.returncode})')

# Verifica config_snapshot ha d0=1e-5 (sanity CLI flow)
import json
cfg = json.load(open(f'checkpoints/{tag_smoke}/config_snapshot.json'))
assert cfg['prodigy_d0'] == 1e-5, f'CLI flow rotto: prodigy_d0={cfg["prodigy_d0"]}'
print(f'  [OK] smoke run ok, config_snapshot.prodigy_d0 = {cfg["prodigy_d0"]}')

# Verifica CSV ha col R27 popolata
import pandas as pd
ldf = pd.read_csv(f'checkpoints/{tag_smoke}/training_log.csv')
assert 'val_T_intra_corr' in ldf.columns, 'R27 col mancante!'
bdf = pd.read_csv(f'checkpoints/{tag_smoke}/training_batch_log.csv')
import math
v = bdf['gn_hidden_fc'].iloc[0]
assert math.isfinite(v), f'LAYER_MAP fix non attivo: gn_hidden_fc={v}'
print(f'  [OK] CSV ha val_T_intra_corr + gn_hidden_fc popolato (R27 fix attivi)')

# Cleanup smoke (best-effort, NFS-safe)
def _robust_rmtree(path, max_retries=3):
    import os, shutil, time
    for attempt in range(max_retries):
        if not os.path.isdir(path): return True
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path): return True
        time.sleep(0.5 * (attempt + 1))
    return not os.path.isdir(path)
_robust_rmtree(f'checkpoints/{tag_smoke}')
print(f'  [OK] smoke cleanup\n\nPRE-FLIGHT passed.')

In [ ]:
# Cell 5 -- SWEEP 5 run con PUSH per-run (pattern R26 + lezione R27: stream subprocess)
import subprocess, sys, time, os, shutil, glob, datetime
import pandas as pd

# Re-define globals (Cell 5 standalone-safe dopo kernel restart)
if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = 'results/Prodigy_Study/ProdigyTuning_R28'
    BRANCH = 'Prodigy_Deep_Study'
    _TMP_MSG = '/tmp/r28_msg.txt' if os.path.isdir('/tmp') else 'r28_msg.txt'
os.makedirs(RESULTS_DIR, exist_ok=True)

SKIP_IF_EXISTS = True   # idempotenza per-run: salta run gia\' completati

def _robust_rmtree(path, max_retries=3):
    for attempt in range(max_retries):
        if not os.path.isdir(path): return True
        shutil.rmtree(path, ignore_errors=True)
        if not os.path.isdir(path): return True
        time.sleep(0.5 * (attempt + 1))
    shutil.rmtree(path, ignore_errors=True)
    return not os.path.isdir(path)

def _build_cli(e):
    cli = [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', str(e['epochs']),
        '--max_steps_per_epoch', str(e['max_steps_per_epoch']),
        '--batch_size', '8', '--val_batch_size', '32',
        '--seq_len', str(e['seq_len']),
        '--cf_hidden_size', str(e['hidden_size']),
        '--cf_rank', str(e['rank']),
        '--cf_max_delay', str(e['max_delay']),
        '--cf_bit_shift', str(e['bit_shift']),
        '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
        '--lambda_bc', '1.0', '--lambda_sr', str(e['lambda_sr']),
        '--lambda_T_aux', str(e['lambda_T_aux']),
        '--scenario_mix', e['scenario_mix'],
        '--cut_in_ratio', str(e['cut_in_ratio']),
        '--noise_scale', '0.0', '--po2_enabled', '1',
        '--n_train', '1500', '--n_val', '300',
        '--max_inf_streak', '99999', '--early_stop_patience', '0',
        '--data_cache', e['cache_path'],
        '--optimizer', e['optimizer'],
        '--lr', str(e['lr']), '--max_lr', str(e['lr']),
        '--scheduler', e['scheduler'],
        '--T0', str(e['T0']),
        '--prodigy_betas', '0.9,0.99',
        '--prodigy_d_coef', str(e['d_coef']),
        '--prodigy_d0', str(e['d0']),
        '--prodigy_weight_decay', '0.01',
        '--prodigy_use_bias_correction', '1',
        '--prodigy_safeguard_warmup', '1',
        '--prodigy_growth_rate', str(e['growth_rate']),
        '--tag', e['tag']]
    return cli

def _dst_for(e):
    return f"{RESULTS_DIR}/{e['axis']}/{e['tag']}"

def _push_run(e):
    tag = e['tag']
    src = f'checkpoints/{tag}'
    dst = _dst_for(e)
    if not os.path.isdir(src):
        print(f'   [WARN push] {src} mancante')
        return False
    _robust_rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')
    val_str = ''
    log_path = f'{dst}/training_log.csv'
    if os.path.isfile(log_path):
        try:
            edf = pd.read_csv(log_path)
            if len(edf) > 0:
                bi = int(edf.val_total.idxmin())
                tc  = edf.get('val_T_tracking_corr', pd.Series([float('nan')])).iloc[bi]
                tci = edf.get('val_T_intra_corr',   pd.Series([float('nan')])).iloc[bi]
                pd_final = edf['prodigy_d'].iloc[-1] if 'prodigy_d' in edf.columns else float('nan')
                val_str = (f'best val={edf.val_total.min():.4f}  T_corr={tc:.3f}  '
                           f'T_intra={tci:.3f}  prodigy_d_final={pd_final:.4f}  '
                           f'(E{bi+1}/{len(edf)})')
        except Exception as ex:
            val_str = f'(log read failed: {ex})'
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (f"results (R28 ProdigyTuning): {tag} ({ts})\n\n{val_str}\n"
           f"desc={e['desc']}\nAxis: {e['axis']}\nBranch: {BRANCH}\n")
    with open(_TMP_MSG, 'w', encoding='utf-8') as fp:
        fp.write(msg)
    try:
        subprocess.run(['git','add',dst], check=True, capture_output=True)
        r = subprocess.run(['git','commit','-F',_TMP_MSG], capture_output=True, text=True)
        if r.returncode != 0:
            if 'nothing to commit' in r.stdout or 'nothing to commit' in r.stderr:
                return True
            print(f'   [push commit fail] {r.stderr[-300:]}'); return False
        subprocess.run(['git','pull','--no-rebase','--no-edit','origin',BRANCH],
                       capture_output=True, text=True)
        r2 = subprocess.run(['git','push','origin',BRANCH], capture_output=True, text=True)
        if r2.returncode != 0:
            print(f'   [push fail] {r2.stderr[-300:]}'); return False
        print(f'   [push OK]')
        return True
    finally:
        try: os.remove(_TMP_MSG)
        except: pass

# Reload EXPERIMENTS se Cell 2 non in memoria (kernel restart safe)
if 'EXPERIMENTS' not in dir():
    raise RuntimeError('EXPERIMENTS non definito: rilancia Cell 2 prima.')

run_results = []
t_start = time.time()
total = len(EXPERIMENTS)

for i, e in enumerate(EXPERIMENTS, 1):
    tag = e['tag']
    dst = _dst_for(e)
    dst_log = f'{dst}/training_log.csv'
    # Idempotenza per-run: skip se gia' completato
    if SKIP_IF_EXISTS and os.path.isfile(dst_log):
        try:
            edf = pd.read_csv(dst_log)
            v_str = f'val={edf.val_total.min():.4f} epochs={len(edf)}/{e["epochs"]}'
            # Considera completo se >= 80% delle epoche pianificate
            if len(edf) >= e['epochs'] * 0.8:
                print(f'\n[{i}/{total}] [SKIP] {tag}: {v_str}')
                run_results.append({'tag': tag, 'axis': e['axis'], 'status':'skipped'})
                continue
            else:
                print(f'\n[{i}/{total}] [PARTIAL] {tag}: {v_str} -> RERUN')
        except Exception:
            print(f'\n[{i}/{total}] [UNREADABLE] {tag} -> RERUN')

    print(f'\n{"="*78}\n[{i}/{total}] {tag}  [axis={e["axis"]}]\n  {e["desc"]}\n{"="*78}')
    t0 = time.time()
    # capture_output=False -> stream stdout al kernel (R27 lesson: evita kernel timeout)
    r = subprocess.run(_build_cli(e), capture_output=False)
    el = time.time() - t0
    status = 'ok' if r.returncode == 0 else f'fail({r.returncode})'
    el_tot = time.time() - t_start
    done_now = sum(1 for x in run_results if x['status']!='skipped') + 1
    eta_min = (el_tot / max(done_now,1)) * (total - i) / 60
    print(f'\n[{i}/{total}] {tag} -> {status}  ({el/60:.1f}min)  ETA={eta_min:.0f}min')
    pushed = _push_run(e)
    run_results.append({'tag': tag, 'axis': e['axis'], 'status': status, 'pushed': pushed})

print(f'\n{"="*78}\nSWEEP DONE in {(time.time()-t_start)/60:.0f}min')
for r in run_results:
    if r['status'] != 'skipped':
        print(f"   {r['tag']:<35} [{r['axis']:<16}] {r['status']:<10} push={r.get('pushed','N/A')}")

In [ ]:
# Cell 6 -- AGGREGATOR + comparison (Prodigy d evolution + T_intra_corr)
import os, glob, json
import pandas as pd, numpy as np
from IPython.display import display, Markdown

# Standalone-safe: re-define globals + re-load EXPERIMENTS se necessario
if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = 'results/Prodigy_Study/ProdigyTuning_R28'
    AGGREGATE_CSV = f'{RESULTS_DIR}/_aggregate_R28.csv'

# Discover all R28 run dirs (auto-discovery anche se EXPERIMENTS non in memoria)
run_dirs = []
for root, _, files in os.walk(RESULTS_DIR):
    if 'training_log.csv' in files and 'config_snapshot.json' in files:
        run_dirs.append(root)
run_dirs = sorted(run_dirs)
print(f'Run R28 discovered: {len(run_dirs)}')

rows = []
for rd in run_dirs:
    cfg = json.load(open(os.path.join(rd, 'config_snapshot.json')))
    edf = pd.read_csv(os.path.join(rd, 'training_log.csv'))
    if len(edf) == 0: continue
    bi = int(edf['val_total'].idxmin())
    row = {
        'tag': cfg['tag'],
        'axis': os.path.basename(os.path.dirname(rd)),
        'n_ep': len(edf),
        'best_ep': bi+1,
        'val_total_best': float(edf['val_total'].iloc[bi]),
        'val_data_best':  float(edf['val_data'].iloc[bi]),
        'T_tracking_corr_best': float(edf.get('val_T_tracking_corr', pd.Series([np.nan])).iloc[bi]),
        'T_intra_corr_best':    float(edf.get('val_T_intra_corr',   pd.Series([np.nan])).iloc[bi]),
        # Prodigy diagnostics (chiave R28)
        'prodigy_d_final':  float(edf['prodigy_d'].iloc[-1]) if 'prodigy_d' in edf.columns else np.nan,
        'prodigy_d_max':    float(edf['prodigy_d_max'].iloc[-1]) if 'prodigy_d_max' in edf.columns else np.nan,
        'prodigy_lr_eff_max':  float(edf['prodigy_lr_eff'].max()) if 'prodigy_lr_eff' in edf.columns else np.nan,
        'prodigy_lr_eff_last': float(edf['prodigy_lr_eff'].iloc[-1]) if 'prodigy_lr_eff' in edf.columns else np.nan,
        # Config
        'd0': float(cfg.get('prodigy_d0', np.nan)),
        'max_steps': int(cfg.get('max_steps_per_epoch', -1)),
        'scheduler': cfg.get('scheduler', '?'),
        # Predicted means per canale (per saturazione check)
        'v0_pred': float(edf.get('val_v0_pred_mean', pd.Series([np.nan])).iloc[bi]),
        'T_pred':  float(edf.get('val_T_pred_mean',  pd.Series([np.nan])).iloc[bi]),
        's0_pred': float(edf.get('val_s0_pred_mean', pd.Series([np.nan])).iloc[bi]),
    }
    rows.append(row)

df = pd.DataFrame(rows).sort_values('T_intra_corr_best', ascending=False)
df.to_csv(AGGREGATE_CSV, index=False)
print(f'Saved aggregate: {AGGREGATE_CSV}')

# ── Tabella sintesi ──
display(Markdown('## R28 sintesi — ordinata per T_intra_corr desc'))
show_cols = ['tag','axis','d0','max_steps','scheduler',
             'val_total_best','val_data_best',
             'T_tracking_corr_best','T_intra_corr_best',
             'prodigy_d_final','prodigy_d_max','prodigy_lr_eff_max','prodigy_lr_eff_last',
             'v0_pred','T_pred','s0_pred']
show_cols = [c for c in show_cols if c in df.columns]
display(df[show_cols].round(4))

# ── Confronto baseline B1 (R28_A0) vs altri ──
if 'R28_A0_baseline_B1' in df['tag'].values:
    base = df[df.tag=='R28_A0_baseline_B1'].iloc[0]
    display(Markdown(f'## Delta vs baseline R28_A0_baseline_B1 (Δ = run − baseline)'))
    delta_rows = []
    for _, r in df.iterrows():
        if r['tag'] == 'R28_A0_baseline_B1': continue
        delta_rows.append({
            'tag': r['tag'],
            'Δ val_data':  r['val_data_best']  - base['val_data_best'],
            'Δ T_tracking': r['T_tracking_corr_best'] - base['T_tracking_corr_best'],
            'Δ T_intra':   r['T_intra_corr_best']    - base['T_intra_corr_best'],
            'd_final ratio': r['prodigy_d_final'] / max(base['prodigy_d_final'], 1e-9),
        })
    display(pd.DataFrame(delta_rows).round(4))

# ── Verdetto per asse ──
display(Markdown('## Verdetti per asse'))
for axis in ['A_d0','B_steps','C_combo','D_warm_restart']:
    sub = df[df.axis == axis]
    if sub.empty: continue
    print(f'\n[{axis}]')
    for _, r in sub.iterrows():
        unlock = '✅ UNLOCK' if r['prodigy_d_final'] > 0.5 else ('⚠ growth ok' if r['prodigy_d_final'] > 0.1 else '❌ stuck')
        lr_alive = '✅' if r['prodigy_lr_eff_last'] > 1e-3 else '❌'
        intra_ok = '✅' if r['T_intra_corr_best'] > 0.1 else ('⚠' if r['T_intra_corr_best'] > 0.05 else '❌')
        print(f'  {r["tag"]:<32}  d_final={r["prodigy_d_final"]:.4f} [{unlock}]  '
              f'lr_eff_last={r["prodigy_lr_eff_last"]:.2e} [{lr_alive}]  '
              f'T_intra={r["T_intra_corr_best"]:.3f} [{intra_ok}]')

In [ ]:
# Cell 7 -- PLOT suite (focus: Prodigy d evolution + lr_eff + T_intra)
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = 'results/Prodigy_Study/ProdigyTuning_R28'
if 'df' not in dir():
    df = pd.read_csv(f'{RESULTS_DIR}/_aggregate_R28.csv')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Plot 1: d evolution epoch-by-epoch per ogni run
ax = axes[0,0]
for _, r in df.iterrows():
    rd_candidates = [d for d in os.listdir(f'{RESULTS_DIR}/{r["axis"]}') if d == r['tag']]
    if not rd_candidates: continue
    log_path = f'{RESULTS_DIR}/{r["axis"]}/{r["tag"]}/training_log.csv'
    if not os.path.isfile(log_path): continue
    edf = pd.read_csv(log_path)
    if 'prodigy_d' not in edf.columns: continue
    ax.plot(edf['epoch'], edf['prodigy_d'], marker='o', label=r['tag'].replace('R28_',''))
ax.set_xlabel('epoch'); ax.set_ylabel('prodigy_d')
ax.set_title('Prodigy d adapter evolution (R28)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_yscale('log')

# Plot 2: lr_eff evolution (lr * d * bias_corr)
ax = axes[0,1]
for _, r in df.iterrows():
    log_path = f'{RESULTS_DIR}/{r["axis"]}/{r["tag"]}/training_log.csv'
    if not os.path.isfile(log_path): continue
    edf = pd.read_csv(log_path)
    if 'prodigy_lr_eff' not in edf.columns: continue
    ax.plot(edf['epoch'], edf['prodigy_lr_eff'], marker='o', label=r['tag'].replace('R28_',''))
ax.set_xlabel('epoch'); ax.set_ylabel('prodigy_lr_eff')
ax.set_title('Effective LR (d × base_lr × bias_corr)')
ax.axhline(1e-3, color='red', linestyle=':', alpha=0.5, label='soglia sano')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_yscale('log')

# Plot 3: T_intra_corr evolution
ax = axes[1,0]
for _, r in df.iterrows():
    log_path = f'{RESULTS_DIR}/{r["axis"]}/{r["tag"]}/training_log.csv'
    if not os.path.isfile(log_path): continue
    edf = pd.read_csv(log_path)
    if 'val_T_intra_corr' not in edf.columns: continue
    ax.plot(edf['epoch'], edf['val_T_intra_corr'], marker='o', label=r['tag'].replace('R28_',''))
ax.set_xlabel('epoch'); ax.set_ylabel('val_T_intra_corr')
ax.set_title('T_intra_corr evolution (R27 metric)')
ax.axhline(0.1, color='orange', linestyle=':', alpha=0.5, label='soglia weak')
ax.axhline(0.3, color='green', linestyle=':', alpha=0.5, label='soglia strong')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Plot 4: scatter d_final vs T_intra_corr (correlazione tra escape e quality)
ax = axes[1,1]
ax.scatter(df['prodigy_d_final'], df['T_intra_corr_best'], s=120, alpha=0.7, edgecolor='black')
for _, r in df.iterrows():
    ax.annotate(r['tag'].replace('R28_','').replace('_baseline_B1','baseline')[:12],
                (r['prodigy_d_final'], r['T_intra_corr_best']),
                fontsize=8, alpha=0.8)
ax.set_xlabel('prodigy_d_final'); ax.set_ylabel('T_intra_corr_best')
ax.set_title('Sblocco d ↔ T_intra: c\'è correlazione?')
ax.set_xscale('log')
ax.grid(alpha=0.3)

plt.tight_layout()
out_png = f'{RESULTS_DIR}/R28_diagnostic_plots.png'
plt.savefig(out_png, dpi=110)
plt.show()
print(f'Salvato: {out_png}')

In [ ]:
# Cell 8 -- Summary + decisione passaggio R29 + push finale aggregator
import os, subprocess, tempfile, pandas as pd
from IPython.display import display, Markdown

if 'RESULTS_DIR' not in dir():
    RESULTS_DIR = 'results/Prodigy_Study/ProdigyTuning_R28'
    AGGREGATE_CSV = f'{RESULTS_DIR}/_aggregate_R28.csv'
    BRANCH = 'Prodigy_Deep_Study'
if 'df' not in dir():
    df = pd.read_csv(AGGREGATE_CSV)

display(Markdown('## Verdetto R28: Prodigy era il bottleneck?'))

any_unlocked = (df['prodigy_d_final'] > 0.5).any()
any_intra_strong = (df['T_intra_corr_best'] > 0.1).any()

best_intra = df.sort_values('T_intra_corr_best', ascending=False).iloc[0]
best_val   = df.sort_values('val_data_best', ascending=True).iloc[0]

print(f'Run con d_final > 0.5 (sblocco Prodigy):      {any_unlocked}')
print(f'Run con T_intra_corr > 0.1 (degenerazione rotta): {any_intra_strong}')
print()
print(f'Best T_intra: {best_intra["tag"]} = {best_intra["T_intra_corr_best"]:.3f}')
print(f'Best val_data: {best_val["tag"]} = {best_val["val_data_best"]:.4f}')
print(f'D1 warm restart: T_intra = '
      f'{df[df.tag=="R28_D1_warm_restart"]["T_intra_corr_best"].values[0]:.3f}' if 'R28_D1_warm_restart' in df['tag'].values else 'D1 mancante')

if any_unlocked and any_intra_strong:
    display(Markdown('### ✅ Prodigy ERA un fattore. Champion R28 → nuovo baseline per R29.'))
elif any_unlocked and not any_intra_strong:
    display(Markdown('### ⚠ Prodigy d sbloccato ma T_intra resta degenere. Prodigy non era sufficient -- continuare con R29.'))
else:
    display(Markdown('### ❌ Prodigy NON era il bottleneck. Procedere a R29 (DecoderFix) per attaccare il rank-collapse.'))

# Push aggregator finale (idempotente)
subprocess.run(['git','add', AGGREGATE_CSV, f'{RESULTS_DIR}/R28_diagnostic_plots.png'], check=False)
msg = f"R28 ProdigyTuning aggregator: {len(df)} run, best T_intra={best_intra['T_intra_corr_best']:.3f}"
with tempfile.NamedTemporaryFile('w', delete=False, suffix='.txt', encoding='utf-8') as fp:
    fp.write(msg); msg_path = fp.name
r = subprocess.run(['git','diff','--cached','--name-only'], capture_output=True, text=True)
if r.stdout.strip():
    subprocess.run(['git','commit','-F',msg_path], check=True)
    subprocess.run(['git','pull','--no-rebase','--no-edit','origin',BRANCH], check=True)
    subprocess.run(['git','push','origin',BRANCH], check=True)
    print('\n[OK] aggregator R28 pushato.')
else:
    print('\n[SKIP] nessuna modifica aggregator da pushare.')
try: os.unlink(msg_path)
except: pass